In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
import pandas as pd
from tensorflow.keras import optimizers

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:

df = pd.DataFrame({
    "soil_moisture": [
        0.10, 0.15, 0.20, 0.25, 0.40, 0.60, 0.35, 0.18,
        0.45, 0.05, 0.08, 0.27, 0.55, 0.70, 0.12, 0.30
    ],
    "temperature": [
        34, 30, 26, 22, 28, 30, 19, 22,
        35, 24, 33, 33, 21, 25, 20, 29
    ],
    "sunlight_hours": [
        9, 8, 7, 4, 8, 10, 3, 10,
        12, 5, 9, 11, 2, 6, 1, 9
    ],
    "needs_water": [
        1, 1, 1, 0, 0, 0, 0, 1,
        1, 0, 0, 1, 0, 0, 1, 1
    ]
})
print(df)
print(df.columns)
X = df[['soil_moisture', 'temperature', 'sunlight_hours']]
y = df['needs_water']
X_min = X.min()
X_max= X.max()
X_scaled = (X-X_min)/(X_max-X_min+1e-8)
opt = optimizers.SGD(learning_rate=0.01,momentum=0.9)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.25, random_state=42
)
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer=opt,loss='binary_crossentropy',
              metrics=['accuracy'])
history = model.fit(
    X_train.values,
    y_train.values,
    validation_data=(X_test.values, y_test.values),
    epochs=100,
    verbose=1,
    batch_size =4
)
history_minbatch = model.fit(X_train,y_train,epochs=100,batch_size=100,verbose=1)



Project OF ANN

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers
import pandas as pd
from tensorflow.keras import optimizers
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import LabelEncoder,StandardScaler
# ---------------------------------------------------------
# Scikit-Learn (Machine Learning utilities)
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split
# Splits the dataset into training and testing sets to evaluate model performance.

from sklearn.preprocessing import LabelEncoder, StandardScaler
# LabelEncoder: Converts categorical labels (e.g., "cat", "dog") into numeric values.
# StandardScaler: Normalizes/standardizes numerical features so they have mean=0 and variance=1.

from sklearn.linear_model import Perceptron
# A simple linear classifier (single-layer neural network). Good for binary classification tasks.

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
# accuracy_score: Measures how many predictions are correct.
# classification_report: Shows precision, recall, and F1-score for each class.
# confusion_matrix: Shows where the model is making correct vs wrong predictions.

# ---------------------------------------------------------
# TensorFlow / Keras (Deep Learning utilities)
# ---------------------------------------------------------
# (Screenshot niche se cut raha hai, lekin aage ka code usually ye hota hai):
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.utils import to_categorical

df = pd.read_csv('datasets/iris_data_species.csv')

df.head()
df['species'].value_counts()



df.info()


sns.pairplot(df,hue='species')



from seaborn._core.typing import ColumnName
X = df.drop(columns=['species',])





label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['species'])
print("Encoded target variable 'y':")
print(y[:5]) # Display first 5 encoded values
print("Original species for first 5 values:")
print(df['species'].head())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
print("First 5 rows of X_scaled:")
print(X_scaled.head())

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


X = df.drop(columns=['species',])
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.25, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(len(np.unique(y)), activation='softmax')
])

model.summary()


model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print("Model compiled successfully with Adam optimizer, sparse_categorical_crossentropy loss, and accuracy metric.")

history = model.fit(X_train, y_train, epochs=50, validation_split=0.2, verbose=0)
print("Model training complete.")

y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
conf_matrix = confusion_matrix(y_test, y_pred)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

import keras_tuner as kt

def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Dense(units=hp.Int('units_1', min_value=32, max_value=128, step=32),
                           activation=hp.Choice('activation_1', ['relu', 'tanh']),
                           input_shape=(X_train.shape[1],)))
    model.add(layers.Dense(units=hp.Int('units_2', min_value=16, max_value=64, step=16),
                           activation=hp.Choice('activation_2', ['relu', 'tanh'])))
    model.add(layers.Dense(len(np.unique(y)), activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Keras Tuner's RandomSearch imported and build_model function defined.")

print("keras-tuner installed successfully.")


def build_model(hp):
    model = keras.Sequential()
    model.add(layers.Dense(units=hp.Int('units_1', min_value=32, max_value=128, step=32),
                           activation=hp.Choice('activation_1', ['relu', 'tanh']),
                           input_shape=(X_train.shape[1],)))
    model.add(layers.Dense(units=hp.Int('units_2', min_value=16, max_value=64, step=16),
                           activation=hp.Choice('activation_2', ['relu', 'tanh'])))
    model.add(layers.Dense(len(np.unique(y)), activation='softmax'))

    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    optimizer = keras.optimizers.Adam(learning_rate=hp_learning_rate)
    
    model.compile(optimizer=optimizer,
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Keras Tuner's RandomSearch imported and build_model function defined.")

tuner = kt.RandomSearch(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='my_dir',
    project_name='iris_hptuner'
)

print("RandomSearch tuner instantiated.")

tuner.search(X_train, y_train, epochs=50, validation_split=0.2)
print("Hyperparameter search initiated.")
